
# Tarea 1 - MLP para clasificar Pokemon legendarios

Este cuaderno resuelve la actividad usando un MLP en PyTorch. La idea central es construir un clasificador binario para predecir la columna `Legendary` a partir de atributos numericos y categoricos del dataset.

La solucion cuida tres puntos importantes:

- El dataset esta desbalanceado: hay muchos mas Pokemon no legendarios que legendarios.
- La columna `Generation` debe tratarse como variable categorica, no como una magnitud continua.
- El archivo esta ordenado por numero/generacion, por lo que el split debe ser aleatorio y estratificado.


In [1]:

from pathlib import Path
import copy
import random

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from models import MODEL_REGISTRY, build_model

SEED = 42
DATA_PATH = Path("data/Pokemon.csv")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device


device(type='cpu')


## 1. Carga y explicacion del dataset

El archivo `Pokemon.csv` contiene 800 registros. Cada fila representa una forma de Pokemon; por eso aparecen casos como megaevoluciones o formas alternativas con el mismo numero de Pokedex.

| Columna | Tipo | Descripcion |
|---|---:|---|
| `#` | Entero | Numero de Pokedex. Es un identificador, no una caracteristica de batalla directa. |
| `Name` | Texto | Nombre o forma del Pokemon. No se usara como predictor porque es texto identificador. |
| `Type 1` | Categorica | Tipo principal, por ejemplo Water, Fire, Grass. |
| `Type 2` | Categorica | Tipo secundario. Tiene valores faltantes cuando el Pokemon no tiene segundo tipo. |
| `Total` | Numerica | Suma de las seis estadisticas base. Es redundante, pero informativa. |
| `HP` | Numerica | Puntos de vida base. |
| `Attack` | Numerica | Ataque fisico base. |
| `Defense` | Numerica | Defensa fisica base. |
| `Sp. Atk` | Numerica | Ataque especial base. |
| `Sp. Def` | Numerica | Defensa especial base. |
| `Speed` | Numerica | Velocidad base. |
| `Generation` | Categorica/ordinal | Generacion en que aparece. Se tratara como categoria para evitar imponer distancias artificiales. |
| `Legendary` | Booleana | Variable objetivo: `True` si el Pokemon es legendario, `False` si no. |


In [2]:

df = pd.read_csv(DATA_PATH)

display(df.head())
print(f"Filas: {df.shape[0]} | Columnas: {df.shape[1]}")
display(pd.DataFrame({"dtype": df.dtypes, "nulos": df.isna().sum(), "unicos": df.nunique()}))


,#,Name,Type 1,Type 2,Total,HP,Attack,Defense,Sp. Atk,Sp. Def,Speed,Generation,Legendary
0,1,Bulbasaur,Grass,Poison,318,45,49,49,65,65,45,1,False
1,2,Ivysaur,Grass,Poison,405,60,62,63,80,80,60,1,False
2,3,Venusaur,Grass,Poison,525,80,82,83,100,100,80,1,False
3,3,VenusaurMega Venusaur,Grass,Poison,625,80,100,123,122,120,80,1,False
4,4,Charmander,Fire,NaN,309,39,52,43,60,50,65,1,False


Filas: 800 | Columnas: 13


,dtype,nulos,unicos
#,int64,0,721
Name,str,0,800
Type 1,str,0,18
Type 2,str,386,18
Total,int64,0,200
HP,int64,0,94
Attack,int64,0,111
Defense,int64,0,103
Sp. Atk,int64,0,105
Sp. Def,int64,0,92


In [3]:

# Resumen de la variable objetivo.
legendary_counts = df["Legendary"].value_counts().rename_axis("Legendary").to_frame("cantidad")
legendary_counts["proporcion"] = legendary_counts["cantidad"] / len(df)
display(legendary_counts)

# Distribucion por generacion: permite ver si la pista de "generacion" importa.
generation_table = pd.crosstab(df["Generation"], df["Legendary"])
generation_table["total"] = generation_table.sum(axis=1)
generation_table["legendary_rate"] = generation_table.get(True, 0) / generation_table["total"]
display(generation_table)


,cantidad,proporcion
Legendary,,
False,735,0.91875
True,65,0.08125


Legendary,False,True,total,legendary_rate
Generation,,,,
1,160,6,166,0.036145
2,101,5,106,0.047170
3,142,18,160,0.112500
4,108,13,121,0.107438
5,150,15,165,0.090909
6,74,8,82,0.097561


In [4]:

# Cuidado: el dataset viene ordenado. Este bloque muestra por que NO conviene separar
# entrenamiento/test usando simplemente las primeras y ultimas filas.
cut = int(len(df) * 0.80)
ordered_train = df.iloc[:cut]
ordered_test = df.iloc[cut:]

print("Generaciones en train si corto por orden:", sorted(ordered_train["Generation"].unique()))
print("Generaciones en test si corto por orden:", sorted(ordered_test["Generation"].unique()))
display(pd.crosstab(ordered_test["Generation"], ordered_test["Legendary"], margins=True))


Generaciones en train si corto por orden: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]
Generaciones en test si corto por orden: [np.int64(5), np.int64(6)]


Legendary,False,True,All
Generation,,,
5,64,14,78
6,74,8,82
All,138,22,160



## 2. Preparacion del dataset

Decisiones tomadas:

- `Type 2`: los nulos se reemplazan por `None`, porque ausencia de segundo tipo tambien es informacion.
- `Legendary`: se transforma a 0/1 para entrenar una clasificacion binaria.
- `Name` y `#`: se excluyen de las entradas. Son identificadores y pueden inducir aprendizaje poco generalizable.
- Variables numericas: se estandarizan usando solo el conjunto de entrenamiento.
- Variables categoricas: `Type 1`, `Type 2` y `Generation` se codifican con one-hot encoding.
- Split: se usa separacion aleatoria estratificada para mantener proporcion similar de legendarios en train, validacion y test.

Sobre `Generation`: la incluyo como categoria porque puede tener informacion real del dataset, pero no la uso como numero continuo. Si el profesor quiere evaluar generalizacion a generaciones futuras no vistas, conviene repetir el experimento con `INCLUDE_GENERATION = False` o con un split por generacion.


In [5]:

df_clean = df.copy()
df_clean["Type 2"] = df_clean["Type 2"].fillna("None")
df_clean["Legendary"] = df_clean["Legendary"].astype(int)

NUMERIC_COLS = ["Total", "HP", "Attack", "Defense", "Sp. Atk", "Sp. Def", "Speed"]
BASE_CATEGORICAL_COLS = ["Type 1", "Type 2"]
INCLUDE_GENERATION = True
CATEGORICAL_COLS = BASE_CATEGORICAL_COLS + (["Generation"] if INCLUDE_GENERATION else [])
TARGET_COL = "Legendary"

print("Numericas:", NUMERIC_COLS)
print("Categoricas:", CATEGORICAL_COLS)


Numericas: ['Total', 'HP', 'Attack', 'Defense', 'Sp. Atk', 'Sp. Def', 'Speed']
Categoricas: ['Type 1', 'Type 2', 'Generation']


In [ ]:
def stratified_split_indices(y, train_size=0.70, val_size=0.15, seed=42):
    """Devuelve indices train/val/test manteniendo la proporcion de clases."""
    rng = np.random.default_rng(seed)
    y = np.asarray(y)
    train_idx, val_idx, test_idx = [], [], []
    for cls in np.unique(y):
        cls_idx = np.where(y == cls)[0]
        rng.shuffle(cls_idx)

        n = len(cls_idx)
        n_train = int(round(n * train_size))
        n_val = int(round(n * val_size))

        train_idx.extend(cls_idx[:n_train])
        val_idx.extend(cls_idx[n_train:n_train + n_val])
        test_idx.extend(cls_idx[n_train + n_val:])

    train_idx = np.array(train_idx)
    val_idx = np.array(val_idx)
    test_idx = np.array(test_idx)
    rng.shuffle(train_idx)
    rng.shuffle(val_idx)
    rng.shuffle(test_idx)
    return train_idx, val_idx, test_idx

train_idx, val_idx, test_idx = stratified_split_indices(df_clean[TARGET_COL], seed=SEED)

train_df = df_clean.iloc[train_idx].copy()
val_df = df_clean.iloc[val_idx].copy()
test_df = df_clean.iloc[test_idx].copy()

split_summary = pd.DataFrame({
    "split": ["train", "val", "test"],
    "filas": [len(train_df), len(val_df), len(test_df)],
    "legendarios": [train_df[TARGET_COL].sum(), val_df[TARGET_COL].sum(), test_df[TARGET_COL].sum()],
})
split_summary["proporcion_legendarios"] = split_summary["legendarios"] / split_summary["filas"]
display(split_summary)

,split,filas,legendarios,proporcion_legendarios
0,train,560,46,0.082143
1,val,120,10,0.083333
2,test,120,9,0.075000


In [7]:

def build_feature_matrices(train_df, val_df, test_df):
    """Ajusta transformaciones en train y las aplica a val/test."""
    mean = train_df[NUMERIC_COLS].mean()
    std = train_df[NUMERIC_COLS].std().replace(0, 1)

    def transform_numeric(part):
        return ((part[NUMERIC_COLS] - mean) / std).reset_index(drop=True)

    def make_categorical(part):
        categorical = part[CATEGORICAL_COLS].copy()
        for col in CATEGORICAL_COLS:
            categorical[col] = categorical[col].astype(str)
        return pd.get_dummies(categorical, columns=CATEGORICAL_COLS, dtype=float)

    train_cat = make_categorical(train_df)
    cat_columns = train_cat.columns

    def transform_categorical(part):
        return make_categorical(part).reindex(columns=cat_columns, fill_value=0).reset_index(drop=True)

    x_train = pd.concat([transform_numeric(train_df), train_cat.reset_index(drop=True)], axis=1).astype("float32")
    x_val = pd.concat([transform_numeric(val_df), transform_categorical(val_df)], axis=1).astype("float32")
    x_test = pd.concat([transform_numeric(test_df), transform_categorical(test_df)], axis=1).astype("float32")

    y_train = train_df[TARGET_COL].to_numpy(dtype="float32")
    y_val = val_df[TARGET_COL].to_numpy(dtype="float32")
    y_test = test_df[TARGET_COL].to_numpy(dtype="float32")

    return x_train, x_val, x_test, y_train, y_val, y_test

x_train, x_val, x_test, y_train, y_val, y_test = build_feature_matrices(train_df, val_df, test_df)

print("Input dim:", x_train.shape[1])
display(x_train.head())


Input dim: 50


,Total,HP,Attack,Defense,Sp. Atk,Sp. Def,Speed,Type 1_Bug,Type 1_Dark,Type 1_Dragon,...,Type 2_Psychic,Type 2_Rock,Type 2_Steel,Type 2_Water,Generation_1,Generation_2,Generation_3,Generation_4,Generation_5,Generation_6
0,0.682278,-0.053428,1.360703,-0.267103,-0.227434,1.533162,0.397816,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
1,-1.886947,-1.897020,-1.182268,-1.848932,-1.123308,-1.302393,-0.274617,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
2,-1.000441,-0.514326,-0.554004,-0.267103,-0.735096,-0.947949,-1.014293,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
3,-1.082525,-0.399102,-0.165079,-0.741652,-1.123308,-0.770727,-1.115158,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
4,-1.476528,-0.936816,-0.733508,-1.089654,-1.123308,-1.160616,-0.879807,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0


In [8]:

def make_loader(x, y, batch_size=32, shuffle=False):
    x_tensor = torch.tensor(x.to_numpy(), dtype=torch.float32)
    y_tensor = torch.tensor(y, dtype=torch.float32)
    dataset = TensorDataset(x_tensor, y_tensor)
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)

BATCH_SIZE = 32
train_loader = make_loader(x_train, y_train, batch_size=BATCH_SIZE, shuffle=True)
val_loader = make_loader(x_val, y_val, batch_size=128, shuffle=False)
test_loader = make_loader(x_test, y_test, batch_size=128, shuffle=False)



## 3. Definicion de modelos

Las arquitecturas estan definidas en `models.py`, como pide el enunciado. Se comparan tres alternativas:

- `shallow`: una capa oculta. Sirve como linea base simple y reduce riesgo de sobreajuste.
- `medium`: dos capas ocultas con `BatchNorm` y `Dropout`. Es el mejor candidato inicial porque el dataset es pequeno.
- `deep`: tres capas ocultas. Tiene mas capacidad, pero tambien mayor riesgo de sobreajuste.

Todas entregan logits, no probabilidades. Eso permite usar `BCEWithLogitsLoss`, que combina sigmoid y binary cross entropy de forma numericamente estable.


In [9]:

input_dim = x_train.shape[1]
for model_name in MODEL_REGISTRY:
    print(f"\n--- {model_name} ---")
    print(build_model(model_name, input_dim))



--- shallow ---
MLPShallow(
  (net): Sequential(
    (0): Linear(in_features=50, out_features=32, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.1, inplace=False)
    (3): Linear(in_features=32, out_features=1, bias=True)
  )
)

--- medium ---
MLPMedium(
  (net): Sequential(
    (0): Linear(in_features=50, out_features=64, bias=True)
    (1): ReLU()
    (2): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (3): Dropout(p=0.2, inplace=False)
    (4): Linear(in_features=64, out_features=32, bias=True)
    (5): ReLU()
    (6): Dropout(p=0.1, inplace=False)
    (7): Linear(in_features=32, out_features=1, bias=True)
  )
)

--- deep ---
MLPDeep(
  (net): Sequential(
    (0): Linear(in_features=50, out_features=128, bias=True)
    (1): ReLU()
    (2): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (3): Dropout(p=0.25, inplace=False)
    (4): Linear(in_features=128, out_features=64, bias=True)
    


## 4. Optimizador y funcion de costo

Como el problema es binario, uso `BCEWithLogitsLoss`. Como la clase positiva es minoritaria, agrego `pos_weight`, que penaliza mas los errores sobre Pokemon legendarios.

El optimizador elegido es `AdamW`: suele funcionar bien en redes pequenas, converge rapido y agrega regularizacion mediante `weight_decay`.


In [10]:

train_pos = float(y_train.sum())
train_neg = float(len(y_train) - y_train.sum())
pos_weight = torch.tensor([train_neg / train_pos], dtype=torch.float32, device=device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

print("Positivos train:", int(train_pos))
print("Negativos train:", int(train_neg))
print("pos_weight:", round(pos_weight.item(), 3))


Positivos train: 46
Negativos train: 514
pos_weight: 11.174



## 5. Entrenamiento y seleccion de arquitectura

Entreno las tres arquitecturas con los mismos hiperparametros para compararlas de forma justa. La seleccion se hace con F1 en validacion, porque la exactitud puede ser enganosa con un dataset tan desbalanceado.


In [11]:

def compute_metrics(y_true, probabilities, threshold=0.5):
    y_true = np.asarray(y_true).astype(int)
    y_pred = (np.asarray(probabilities) >= threshold).astype(int)

    tp = int(((y_pred == 1) & (y_true == 1)).sum())
    tn = int(((y_pred == 0) & (y_true == 0)).sum())
    fp = int(((y_pred == 1) & (y_true == 0)).sum())
    fn = int(((y_pred == 0) & (y_true == 1)).sum())

    accuracy = (tp + tn) / max(tp + tn + fp + fn, 1)
    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    f1 = 2 * precision * recall / max(precision + recall, 1e-12)

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "tp": tp,
        "tn": tn,
        "fp": fp,
        "fn": fn,
    }


def predict_probabilities(model, loader):
    model.eval()
    probabilities, targets = [], []
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            logits = model(xb)
            probabilities.append(torch.sigmoid(logits).cpu().numpy())
            targets.append(yb.numpy())
    return np.concatenate(probabilities), np.concatenate(targets)


def evaluate_loss(model, loader):
    model.eval()
    losses = []
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            yb = yb.to(device)
            logits = model(xb)
            losses.append(criterion(logits, yb).item() * len(yb))
    return sum(losses) / len(loader.dataset)


def best_threshold_for_f1(y_true, probabilities):
    best_threshold = 0.5
    best_metrics = compute_metrics(y_true, probabilities, threshold=best_threshold)
    for threshold in np.linspace(0.05, 0.95, 91):
        metrics = compute_metrics(y_true, probabilities, threshold=threshold)
        if metrics["f1"] > best_metrics["f1"]:
            best_threshold = float(threshold)
            best_metrics = metrics
    return best_threshold, best_metrics


In [12]:

def train_model(model_name, epochs=250, lr=1e-3, weight_decay=1e-4, patience=40):
    model = build_model(model_name, input_dim).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

    best_state = copy.deepcopy(model.state_dict())
    best_val_f1 = -1.0
    best_val_loss = float("inf")
    epochs_without_improvement = 0
    history = []

    for epoch in range(1, epochs + 1):
        model.train()
        train_loss_sum = 0.0
        for xb, yb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)

            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()

            train_loss_sum += loss.item() * len(yb)

        train_loss = train_loss_sum / len(train_loader.dataset)
        val_loss = evaluate_loss(model, val_loader)
        val_probs, val_targets = predict_probabilities(model, val_loader)
        val_metrics = compute_metrics(val_targets, val_probs, threshold=0.5)

        history.append({
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "val_f1_05": val_metrics["f1"],
            "val_recall_05": val_metrics["recall"],
            "val_precision_05": val_metrics["precision"],
        })

        improved = (val_metrics["f1"] > best_val_f1) or (
            val_metrics["f1"] == best_val_f1 and val_loss < best_val_loss
        )
        if improved:
            best_val_f1 = val_metrics["f1"]
            best_val_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= patience:
            break

    model.load_state_dict(best_state)
    val_probs, val_targets = predict_probabilities(model, val_loader)
    threshold, tuned_metrics = best_threshold_for_f1(val_targets, val_probs)

    return {
        "model": model,
        "history": pd.DataFrame(history),
        "threshold": threshold,
        "val_metrics": tuned_metrics,
        "epochs_trained": len(history),
    }

results = {}
for name in MODEL_REGISTRY:
    print(f"Entrenando {name}...")
    results[name] = train_model(name)

summary_rows = []
for name, result in results.items():
    row = {"model": name, "threshold": result["threshold"], "epochs": result["epochs_trained"]}
    row.update({f"val_{k}": v for k, v in result["val_metrics"].items() if k in ["accuracy", "precision", "recall", "f1", "tp", "tn", "fp", "fn"]})
    summary_rows.append(row)

summary = pd.DataFrame(summary_rows).sort_values(["val_f1", "val_recall"], ascending=False)
display(summary)

best_name = summary.iloc[0]["model"]
best_model = results[best_name]["model"]
best_threshold = float(results[best_name]["threshold"])
print(f"Mejor arquitectura: {best_name} | threshold: {best_threshold:.2f}")


Entrenando shallow...
Entrenando medium...
Entrenando deep...


,model,threshold,epochs,val_accuracy,val_precision,val_recall,val_f1,val_tp,val_tn,val_fp,val_fn
2,deep,0.50,133,0.983333,0.900000,0.9,0.900000,9,109,1,1
1,medium,0.50,99,0.958333,0.666667,1.0,0.800000,10,105,5,0
0,shallow,0.71,65,0.933333,0.562500,0.9,0.692308,9,103,7,1


Mejor arquitectura: deep | threshold: 0.50



## 6. Evaluacion del modelo elegido

Uso el conjunto de test solo al final. Reporto exactitud, precision, recall, F1 y matriz de confusion.

- La exactitud mide aciertos totales, pero puede verse artificialmente alta por el desbalance.
- La precision responde: de los que predije como legendarios, cuantos realmente lo eran.
- El recall responde: de los legendarios reales, cuantos encontre.
- F1 combina precision y recall, por eso es una metrica razonable para elegir modelo.


In [13]:

test_probs, test_targets = predict_probabilities(best_model, test_loader)
test_metrics = compute_metrics(test_targets, test_probs, threshold=best_threshold)

display(pd.DataFrame([test_metrics]).T.rename(columns={0: "valor"}))

confusion = pd.DataFrame(
    [[test_metrics["tn"], test_metrics["fp"]], [test_metrics["fn"], test_metrics["tp"]]],
    index=["Real no legendario", "Real legendario"],
    columns=["Pred no legendario", "Pred legendario"],
)
display(confusion)


,valor
accuracy,0.950000
precision,0.714286
recall,0.555556
f1,0.625000
tp,5.000000
tn,109.000000
fp,2.000000
fn,4.000000


,Pred no legendario,Pred legendario
Real no legendario,109,2
Real legendario,4,5


In [14]:

test_results = test_df.copy().reset_index(drop=True)
test_results["prob_legendary"] = test_probs
test_results["predicted_legendary"] = (test_probs >= best_threshold).astype(int)
test_results["error"] = test_results[TARGET_COL] != test_results["predicted_legendary"]

cols_to_show = [
    "#", "Name", "Type 1", "Type 2", "Generation", "Total", "HP", "Attack", "Defense",
    "Sp. Atk", "Sp. Def", "Speed", "Legendary", "prob_legendary", "predicted_legendary"
]

misclassified = test_results[test_results["error"]].copy()
print(f"Casos mal clasificados: {len(misclassified)}")
display(misclassified[cols_to_show].sort_values("prob_legendary", ascending=False).head(10))


Casos mal clasificados: 6


,#,Name,Type 1,Type 2,Generation,Total,HP,Attack,Defense,Sp. Atk,Sp. Def,Speed,Legendary,prob_legendary,predicted_legendary
114,488,Cresselia,Psychic,None,4,600,120,70,120,75,130,85,0,0.996424,1
33,469,Yanmega,Bug,Flying,4,515,86,76,86,116,56,95,0,0.741913,1
99,250,Ho-oh,Fire,Flying,2,680,106,130,90,110,154,90,1,0.111257,0
80,386,DeoxysSpeed Forme,Psychic,None,3,600,50,95,90,95,90,180,1,0.002722,0
75,639,Terrakion,Rock,Fighting,5,580,91,129,90,72,90,108,1,0.000058,0
94,640,Virizion,Grass,Fighting,5,580,91,90,72,90,129,108,1,0.000003,0



## 7. Respuestas finales

### 1. Interpretacion de la matriz de confusion

La matriz de confusion separa cuatro casos:

- **TN**: Pokemon no legendarios correctamente clasificados como no legendarios.
- **TP**: Pokemon legendarios correctamente clasificados como legendarios.
- **FP**: Pokemon no legendarios que el modelo marco como legendarios. Es un falso aviso; suelen ser Pokemon con estadisticas altas o perfiles parecidos a legendarios.
- **FN**: Pokemon legendarios que el modelo no detecto. Este error es mas grave si el objetivo es encontrar legendarios, porque el modelo deja pasar un caso positivo real.

Si tuviera que elegir Pokemon para mi equipo desde los errores, preferiria mirar primero los **FN**, porque son legendarios reales que el modelo subestimo. Los **FP** tambien pueden ser interesantes si tienen estadisticas muy altas, pero no cumplen la condicion de ser legendarios.

### 2. Caso mal clasificado

El bloque anterior lista los casos mal clasificados. Para interpretar uno, conviene mirar su probabilidad, generacion, tipos y estadisticas. Una explicacion esperable es que el modelo se equivoque cuando un no legendario tiene estadisticas muy altas, o cuando un legendario tiene valores parecidos a Pokemon normales. Tambien puede ocurrir por el bajo numero de ejemplos legendarios: solo hay 65 en todo el dataset.

### 3. Mayor desafio

El mayor desafio fue el desbalance de clases y la preparacion correcta del dataset. Si se usa accuracy solamente, un modelo que predice casi todo como no legendario puede parecer bueno. Para solucionarlo use split estratificado, `pos_weight` en la funcion de perdida, F1/recall como metricas y one-hot encoding para las variables categoricas.


In [15]:

if len(misclassified) > 0:
    case = misclassified.sort_values("prob_legendary", ascending=False).iloc[0]
    real = "legendario" if case["Legendary"] == 1 else "no legendario"
    pred = "legendario" if case["predicted_legendary"] == 1 else "no legendario"
    print("Ejemplo para comentar en la respuesta:")
    print(f"Pokemon: {case['Name']}")
    print(f"Real: {real} | Predicho: {pred} | Probabilidad legendario: {case['prob_legendary']:.3f}")
    print(f"Tipos: {case['Type 1']} / {case['Type 2']} | Generacion: {case['Generation']} | Total: {case['Total']}")
else:
    print("No hubo errores en test con esta ejecucion. En ese caso, comenta que el split produjo un test sin errores y revisa validacion para encontrar un caso dificil.")


Ejemplo para comentar en la respuesta:
Pokemon: Cresselia
Real: no legendario | Predicho: legendario | Probabilidad legendario: 0.996
Tipos: Psychic / None | Generacion: 4 | Total: 600



## 8. IA Generativa

1. **Si.** Utilice una herramienta de IA Generativa como apoyo para estructurar la solucion, revisar el flujo de preprocesamiento, proponer arquitecturas MLP y redactar explicaciones.
2. La utilice principalmente en la preparacion del dataset, la definicion del pipeline de entrenamiento/evaluacion, la interpretacion de metricas y la redaccion de respuestas. Los resultados numericos deben obtenerse ejecutando el notebook y revisando que el codigo funcione correctamente en el entorno local.

## Nota sobre la pista de "generacion"

El profesor probablemente se referia a la columna `Generation`. Esa columna puede causar problemas por tres razones:

- Si se usa como numero continuo, el modelo podria asumir que la distancia entre generacion 1 y 2 significa lo mismo que entre 5 y 6. Para este caso es mas razonable usar one-hot encoding.
- Como el CSV esta ordenado por Pokedex/generacion, un split sin shuffle puede dejar generaciones distintas en train y test, produciendo una evaluacion poco representativa.
- La proporcion de legendarios cambia por generacion, asi que la variable puede ser predictiva, pero tambien puede introducir dependencia del periodo/generacion del dataset.

Tambien podria haber una segunda lectura: al final del enunciado aparece una seccion de **IA Generativa**, donde piden declarar si se uso IA. Como este cuaderno fue preparado con asistencia de IA, conviene responder esa seccion de forma transparente.
